In [ ]:
!pip install newspaper3k lxml_html_clean nltk nrclex requests transformers torch -q
!pip install --upgrade transformers -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 55.4 MB/s eta 0:00:00


In [ ]:
import requests
import warnings
import nltk
from collections import defaultdict

warnings.filterwarnings("ignore")

nltk.download("punkt",                       quiet=True)
nltk.download("punkt_tab",                   quiet=True)
nltk.download("stopwords",                   quiet=True)
nltk.download("vader_lexicon",               quiet=True)
nltk.download("movie_reviews",               quiet=True)
nltk.download("averaged_perceptron_tagger",  quiet=True)

print(" All imports and NLTK resources ready.")

 All imports and NLTK resources ready.


In [ ]:
from newspaper import Article

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
    "Accept":          "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Referer":         "https://www.google.com/",
    "DNT":             "1",
}

URL = (
    "https://venturebeat.com/ai/sam-altman-at-ted-2025-inside-the-most-"
    "uncomfortable-and-important-ai-interview-of-the-year/"
)

# Fetch HTML with requests
response = requests.get(URL, headers=HEADERS, timeout=15)
print(f"HTTP Status  : {response.status_code}")
print(f"Content Size : {len(response.text):,} characters")

# Inject HTML into newspaper3k (bypasses .download())
article = Article(URL)
article.set_html(response.text)
article.parse()
article.nlp()

print("\n" + "=" * 60)
print("ARTICLE EXTRACTED SUCCESSFULLY")
print("=" * 60)
print(f"Title   : {article.title}")
print(f"Authors : {article.authors}")
print(f"Date    : {article.publish_date}")
print(f"\nFull Text Length: {len(article.text):,} characters")
print(f"\n── First 600 characters of article text ──")
print(article.text[:600])

HTTP Status  : 200
Content Size : 129,233 characters

ARTICLE EXTRACTED SUCCESSFULLY
Title   : Sam Altman at TED 2025: Inside the most uncomfortable — and important — AI interview of the year
Authors : []
Date    : 2025-04-15 21:17:23+00:00

Full Text Length: 8,697 characters

── First 600 characters of article text ──
OpenAI CEO Sam Altman revealed that his company has grown to 800 million weekly active users and is experiencing "unbelievable" growth rates, during a sometimes tense interview at the TED 2025 conference in Vancouver last week.

"I have never seen growth in any company, one that I've been involved with or not, like this," Altman told TED head Chris Anderson during their on-stage conversation. "The growth of ChatGPT — it is really fun. I feel deeply honored. But it is crazy to live through, and our teams are exhausted and stressed."

The interview, which closed out the final day of TED 2025: Hu


In [ ]:
from transformers import pipeline, BartForConditionalGeneration, BartTokenizer

print("Loading BART summarization model...")

# Load model and tokenizer directly — bypasses the task registry entirely
model_name = "facebook/bart-large-cnn"
tokenizer  = BartTokenizer.from_pretrained(model_name)
model      = BartForConditionalGeneration.from_pretrained(model_name)

# Tokenize the article text (BART max input = 1024 tokens)
text_chunk = article.text[:3000]
inputs     = tokenizer(
    text_chunk,
    return_tensors = "pt",
    max_length     = 1024,
    truncation     = True
)

# Generate summary
summary_ids = model.generate(
    inputs["input_ids"],
    max_length     = 200,
    min_length     = 100,
    length_penalty = 2.0,
    num_beams      = 4,
    early_stopping = True
)

llm_summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print("\n" + "=" * 60)
print("Q1 — LLM SUMMARY  (BART-large-CNN)")
print("=" * 60)
print(llm_summary)

Loading BART summarization model...


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]


Q1 — LLM SUMMARY  (BART-large-CNN)
OpenAI CEO Sam Altman says his company is experiencing "unbelievable" growth rates. Altman spoke at TED 2025: Humanity Reimagined conference in Vancouver last week. The company recently closed a $40 billion funding round, valuing it at $300 billion. OpenAI is reportedly considering launching its own social network to compete with Elon Musk's X, though Altman neither confirmed nor denied these reports during the TED interview. The interview showcased the increasing scrutiny the company faces as its technology transforms society at a pace that alarms even some of its supporters.


In [ ]:
from transformers import pipeline

# zero-shot-classification
zero_shot = pipeline(
    "zero-shot-classification",
    model = "facebook/bart-large-mnli"
)

# LLM Sentiment
llm_sentiment_result = zero_shot(
    article.text[:1000],
    candidate_labels = ["positive", "negative", "neutral", "mixed"],
    multi_label      = False
)
llm_sentiment_label = llm_sentiment_result["labels"][0]

print("=" * 60)
print("Q2 — LLM SENTIMENT  (Zero-Shot BART-large-MNLI)")
print("=" * 60)
for label, score in zip(llm_sentiment_result["labels"],
                        llm_sentiment_result["scores"]):
    bar = "█" * int(score * 40)
    print(f"  {label:<12} {score:.4f}  {bar}")
print(f"\n  → LLM Sentiment Label: {llm_sentiment_label.upper()}")

# LLM Main Theme
candidate_themes = [
    "AI safety and ethics",
    "OpenAI business transformation",
    "Artificial General Intelligence (AGI)",
    "Impact of AI on society and humanity",
    "AI regulation and governance",
    "Artist rights and intellectual property",
    "Corporate power and accountability",
    "Autonomous AI agents",
    "Content moderation policy",
    "AI-driven economic growth",
]

theme_result = zero_shot(
    article.text[:1000],
    candidate_labels = candidate_themes,
    multi_label      = True
)

print("\n" + "=" * 60)
print("Q4 — MAIN THEME  (Zero-Shot BART-large-MNLI)")
print("=" * 60)
for label, score in zip(theme_result["labels"], theme_result["scores"]):
    bar = "█" * int(score * 40)
    print(f"  {label:<45} {score:.4f}  {bar}")

main_theme = theme_result["labels"][0]
print(f"\n  → Main Theme: {main_theme}")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Q2 — LLM SENTIMENT  (Zero-Shot BART-large-MNLI)
  mixed        0.8118  ████████████████████████████████
  negative     0.1154  ████
  positive     0.0553  ██
  neutral      0.0175  

  → LLM Sentiment Label: MIXED

Q4 — MAIN THEME  (Zero-Shot BART-large-MNLI)
  Impact of AI on society and humanity          0.9358  █████████████████████████████████████
  OpenAI business transformation                0.8074  ████████████████████████████████
  AI safety and ethics                          0.7731  ██████████████████████████████
  Autonomous AI agents                          0.7664  ██████████████████████████████
  AI-driven economic growth                     0.6412  █████████████████████████
  Corporate power and accountability            0.6004  ████████████████████████
  AI regulation and governance                  0.5929  ███████████████████████
  Artificial General Intelligence (AGI)         0.5551  ██████████████████████
  Content moderation policy                     0.4731  █████

In [ ]:
from transformers import pipeline

# text-classification IS supported
print("Loading emotion classifier...")
emotion_pipe = pipeline(
    "text-classification",
    model  = "j-hartmann/emotion-english-distilroberta-base",
    top_k  = None
)

from nltk.tokenize import sent_tokenize
from collections import defaultdict

sentences = sent_tokenize(article.text)

# Chunk into ~400 char segments
chunks, current = [], ""
for sent in sentences:
    if len(current) + len(sent) < 400:
        current += " " + sent
    else:
        chunks.append(current.strip())
        current = sent
if current:
    chunks.append(current.strip())

print(f"Analyzing {len(chunks)} text chunks...")

emotion_totals = defaultdict(float)
for chunk in chunks:
    results = emotion_pipe(chunk[:512])[0]
    for r in results:
        emotion_totals[r["label"]] += r["score"]

emotion_avg      = {k: v / len(chunks) for k, v in emotion_totals.items()}
sorted_emotions  = sorted(emotion_avg.items(), key=lambda x: x[1], reverse=True)
dominant_emotion = sorted_emotions[0][0]

print("\n" + "=" * 60)
print("Q3 — GENERAL EMOTION  (DistilRoBERTa)")
print("=" * 60)
for emotion, score in sorted_emotions:
    bar = "█" * int(score * 40)
    print(f"  {emotion:<12} {score:.4f}  {bar}")
print(f"\n  → Dominant Emotion: {dominant_emotion.upper()}")

Loading emotion classifier...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: j-hartmann/emotion-english-distilroberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Analyzing 30 text chunks...

Q3 — GENERAL EMOTION  (DistilRoBERTa)
  neutral      0.6435  █████████████████████████
  fear         0.1042  ████
  surprise     0.0999  ███
  joy          0.0696  ██
  anger        0.0386  █
  disgust      0.0271  █
  sadness      0.0171  

  → Dominant Emotion: NEUTRAL
